# C3 exp09 on v3: three examples per label

Qwen2.5-7B-Instruct (4-bit), k = 0, definitions on, **data v3**: exp08's best setting with one change, **3 solved train examples per label** (21) instead of 2. Baseline: exp08's `c3_exp08_7b_shots2` (2 per label, macro-F1 0.341 on v3).

Two runs, one per T4, at the same time:
- **shots3**: 3 per label, seed 0, label order: the comparison with exp08's 2 per label;
- **shots3_seed1**: the same with seed 1 (other examples): how much the draw of examples alone moves the score. It uses the second GPU, so it adds no waiting time.

Not calibrated: exp08 found calibration does not help in either form.

Speed as in exp08: `--prefix-cache` (checked first in section 5; without it the runs go at batch 16), batch 32, both GPUs, the model downloaded once.

Note: `tasks/C3-prompting-baseline/experiments/exp09_three-shots-7b/NOTE.md`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v3-meetings` dataset attached (only that one: the data cell looks for v3).

Expected time: about 30 minutes with the cache, about 4 hours without it (estimates; the first progress lines of each log show the real speed).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` is needed for the 4-bit 7B model. Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes

## 3. Data folder: the v3 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v3"]
assert len(hits) == 1, f"expected exactly one v3 dataset under /kaggle/input, found {hits} : is smi-v3-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Download the model once
Both GPUs load the same weights: downloading them here first means neither run waits on the other's download.

In [ ]:
from huggingface_hub import snapshot_download

SEVEN_B = "Qwen/Qwen2.5-7B-Instruct"
print(snapshot_download(SEVEN_B))

## 5. Check the prefix cache on the 7B (~3 min)
16 val segments in exp09's setting (3 examples per label): the 8 longest are scored with the cache, as a padded batch, and one at a time (the reference). The result decides the runs' flags below.

In [ ]:
import shutil

shutil.rmtree("/kaggle/working/checks", ignore_errors=True)  # a throwaway check : always measured afresh
check = subprocess.run(
    ["python", "scripts/trial_prompting.py", "--data-root", DATA_ROOT, "--out-dir", "/kaggle/working/checks",
     "--run-name", "prefix_cache_check", "--model", SEVEN_B, "--load-in-4bit", "--prefix-cache",
     "--shots-per-class", "3", "--shot-seed", "0", "--limit", "16", "--batch-size", "16"],
    cwd="/kaggle/working/repo", env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONPATH="src"),
    capture_output=True, text=True)
print("\n".join(l for l in check.stdout.splitlines() if l.startswith("prefix cache")))
if check.returncode != 0:
    print("\n".join(check.stderr.strip().splitlines()[-3:]))

USE_CACHE = check.returncode == 0
SPEED_FLAGS = ["--prefix-cache", "--batch-size", "32"] if USE_CACHE else ["--batch-size", "16"]
print("USE_CACHE =", USE_CACHE, SPEED_FLAGS, "" if USE_CACHE else ": the check failed , the runs go without the cache (~4 h)")

## 6. Helpers: a queue of runs on one GPU, and waiting while showing progress

In [ ]:
def queue(gpu, jobs):
    """run the jobs one after another on one GPU , in the background ; each run logs to logs/<run_name>.log"""
    commands = []
    for run_name, extra in jobs:
        commands.append(
            f"python scripts/trial_prompting.py --data-root '{DATA_ROOT}' --out-dir /kaggle/working/runs "
            f"--run-name {run_name} --model {SEVEN_B} --load-in-4bit {' '.join(SPEED_FLAGS + extra)} "
            f"> /kaggle/working/logs/{run_name}.log 2>&1"
        )
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    # "; " : a failed run does not stop the next one in the queue
    return subprocess.Popen("; ".join(commands), shell=True, cwd="/kaggle/working/repo", env=env)

def last_line(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """every 2 minutes , print the latest line of every log , until all queues have finished"""
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        for path in sorted(glob.glob("/kaggle/working/logs/*.log")):
            print(f"{os.path.basename(path)[:-4]:26} {last_line(path)[-80:]}")
        print()
    print("all queues finished")

## 7. Run both GPUs
If the session stops, run this cell again: finished runs are skipped and an interrupted run continues where it stopped.
With the cache, each log starts with a `prefix cache :` line: each run repeats the check on its own longest prompts.

In [ ]:
RUNS = ["c3_exp09_7b_shots3", "c3_exp09_7b_shots3_seed1"]
wait([
    queue(0, [("c3_exp09_7b_shots3", ["--shots-per-class", "3", "--shot-seed", "0"])]),
    queue(1, [("c3_exp09_7b_shots3_seed1", ["--shots-per-class", "3", "--shot-seed", "1"])]),
])

## 8. Results
Baseline (exp08, v3): 2 per label 0.341; 1 per label 0.271. Prompt noise (exp07): differences below ~0.035 are not real.

In [ ]:
print(f"{'run':28} {'macro-F1':>8} {'other F1':>8} {'pred. other':>11}  (gold other 13,634)")
for name in RUNS:
    path = f"/kaggle/working/runs/{name}"
    if not os.path.exists(f"{path}/metrics.json"):
        print(f"{name:28} (missing : see logs/{name}.log)"); continue
    m = json.load(open(f"{path}/metrics.json"))
    predicted_other = sum(json.loads(line)["label"] == "other" for line in open(f"{path}/predictions.jsonl"))
    print(f"{name:28} {m['primary']['macro_f1_6']:8.4f} {m['per_class']['other']['f1']:8.3f} {predicted_other:11,}")
    print(" " * 28, {k: round(v["f1"], 3) for k, v in m["per_class"].items()})
    print(" " * 28, "prefix cache:", m["prompting"].get("prefix_cache"))

## 9. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp09_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp09_runs.zip runs logs
!ls -la c3_exp09_runs.zip